# Brasa — pré-treino, SFT e avaliação (GPU do Colab) · notebook v5
Treina um modelo **próprio, sem pesos externos**, com `python/model.py` do projeto. Escolha *Ambiente de execução → Alterar tipo → GPU* (A100/L4 ideal; T4 serve para o preset `small`).
Tudo é salvo no Drive e **retomável**: se a sessão cair, rode as células de novo.

**Sempre abra a versão mais nova deste notebook** pelo GitHub — [abrir no Colab](https://colab.research.google.com/github/va415988-glitch/Brasa-IA/blob/main/pretrain/brasa_pretrain.ipynb) ou *Arquivo → Abrir notebook → GitHub → va415988-glitch/Brasa-IA → pretrain/brasa_pretrain.ipynb* (marque “incluir repositórios privados” se o repositório for privado).
O `git pull` da célula seguinte atualiza os scripts, mas não as células deste notebook; ela avisa quando ele estiver desatualizado.

**O Colab encerra o servidor depois de ~90 min sem interação** — rodar código pela extensão do VS Code não conta como interação
(na rodada de 03/10 o treino morreu no passo ~2950, aos 89 min). Por isso o treino roda em **blocos de 78 min**: a célula para
sozinha e salva; rode-a de novo para continuar. Se o servidor cair mesmo assim, rode as células 1, 2, 3 e a do treino de novo:
ele retoma do último checkpoint no Drive (salvo a cada 250 passos, ~7 min).

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
# Dados e checkpoints ficam no Google Drive: sem ele, tudo se perde quando a sessão acaba.
try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception as error:
    print('Não foi possível montar o Google Drive aqui:', type(error).__name__, error)
    print('No VS Code: atualize a extensão Colab (o suporte a drive.mount é recente). Se continuar falhando,')
    print('rode este notebook pelo navegador: https://colab.research.google.com/github/va415988-glitch/Brasa-IA/blob/main/pretrain/brasa_pretrain.ipynb')
    raise
WORK = '/content/drive/MyDrive/brasa-pretrain'
!mkdir -p {WORK}

In [ ]:
NOTEBOOK_VERSION = 5
# Repositório privado? Gere um token (GitHub > Settings > Developer settings) e preencha.
GITHUB_TOKEN = ''
import os
url = 'github.com/va415988-glitch/Brasa-IA.git'
auth = f'{GITHUB_TOKEN}@' if GITHUB_TOKEN else ''
if not os.path.exists('/content/Brasa-IA'):
    !git clone https://{auth}{url} /content/Brasa-IA
else:
    !git -C /content/Brasa-IA pull --ff-only
%cd /content/Brasa-IA
!git log --oneline -1
!pip -q install datasets tokenizers numpy pytest
latest = open('pretrain/colab_version.txt').read().strip() if os.path.exists('pretrain/colab_version.txt') else '?'
if latest == '?':
    print('⚠️  O código baixado ainda não tem a versão deste notebook: faça o git push no computador e rode esta célula de novo.')
elif str(NOTEBOOK_VERSION) != latest:
    print(f'⚠️  Este notebook (v{NOTEBOOK_VERSION}) está desatualizado: o repositório já tem a v{latest}.')
    print('   Reabra pelo GitHub (link na primeira célula) antes de continuar.')
else:
    print(f'✓ Notebook v{NOTEBOOK_VERSION} em dia com o repositório.')

In [ ]:
# 1) Dados abertos + tokenizer próprio. Demora (download). SCALE=1 ~ 7 GB de texto (~1,5 bi de tokens).
# Se {WORK}/data já existe (rodada anterior), tudo é verificado e pulado em poucos minutos.
SCALE = 1.0
!python pretrain/prepare_data.py --out {WORK}/data --scale {SCALE}

### Verificação antes de gastar horas de GPU (~3 min)
Roda os testes do pipeline e um treino-relâmpago do preset `tiny` nos dados reais. Se algo do ambiente estiver errado
(versão do torch sem `Muon`, memória, dados), aparece aqui e não depois de horas. Tudo verde? Siga para o treino.

In [ ]:
import torch
print('torch', torch.__version__, '| GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'nenhuma',
      '| Muon:', hasattr(torch.optim, 'Muon'), '| bf16:', torch.cuda.is_available() and torch.cuda.is_bf16_supported())
!python -m pytest -q tests/test_model_v2.py tests/test_pretrain_sft.py tests/test_build_agent_sft.py tests/test_build_code_sft.py
!rm -rf /content/smoke && python pretrain/train.py --data {WORK}/data --out /content/smoke --preset tiny --tokens 4e6 --batch-tokens 65536 --micro-batch 16 --warmup-steps 10 --eval-every 30 --eval-batches 2 --save-every 100000 | tail -5

In [ ]:
# 2) Treino do zero. PRESET: 'base' (~100M, 2 bi de tokens = 7636 passos; ~3h50 de A100 a ~144 mil tok/s), 'small' (~40M, T4) ou 'large'.
# Roda em blocos de 78 min (MAX_HOURS=1.3): o Colab derruba o servidor após ~90 min sem interação. Quando a célula
# terminar com "limite de tempo atingido", rode-a de novo: retoma do last.pt. Falta de memória? Reduza MICRO (L4: 8).
# A receita atual (Muon + WSD + QK-norm) não retoma estados da receita antiga: por isso a pasta leva RUN.
PRESET, MICRO, MAX_HOURS, RUN = 'base', 16, 1.3, 'v2'
!python pretrain/train.py --data {WORK}/data --out {WORK}/out-{PRESET}-{RUN} --preset {PRESET} --micro-batch {MICRO} --max-hours {MAX_HOURS} --save-every 250 --compile

## 3) SFT — conversar, agir como agente e responder no formato do produto
Mistura, com a origem de cada parte registrada no checkpoint:
- **decisões de agente** (`datasets/agent_sft_v1`): o protocolo de `cognitive_dialogue.py` com ferramentas reais do runtime sobre 105 repositórios abertos, pesquisa com artigos reais da Wikipédia, respostas humanas (OpenAssistant 2, Dolly) e diálogos escritos por LLM (rotulados `authored-llm-v1`);
- **conversa**: OpenAssistant 2 e Dolly completos, conversas do projeto e os diálogos escritos, como chat;
- **código**: problemas do MBPP no contrato de plano JSON da bancada;
- **disciplina de evidência**: `cognitive_sft_v2` e `cognitive_alignment_v4`.
`--replay 0.2` mistura texto de pré-treino para reduzir o esquecimento (no `small`, o SFT piorou a perplexidade geral de 41,5 para 56,2).

In [ ]:
BASE = f'{WORK}/out-{PRESET}-{RUN}/best.safetensors'  # o modelo da célula 2 ou 2b, a que você rodou por último
!python pretrain/build_code_sft.py --out {WORK}/data/sft_code_plans.jsonl
LOCAL = ','.join(['python/data/*.jsonl', f'{WORK}/data/sft_code_plans.jsonl',
                  'datasets/agent_sft_v1/train.jsonl.gz', 'datasets/agent_sft_v1/authored_chat.jsonl.gz',
                  'datasets/cognitive_sft_v2/train.jsonl', 'datasets/cognitive_alignment_v4/train.jsonl'])
!python pretrain/sft.py --data {WORK}/data --base {BASE} --out {WORK}/sft-{PRESET}-{RUN} --hf oasst2,dolly --epochs 3 --replay 0.2 --local "{LOCAL}"

## 4) Avaliar o agente aqui mesmo, na GPU (~5 min)
Mede o SFT no held-out de `datasets/agent_sft_v1` (repositórios e artigos que o treino não viu), gerando direto no modelo
com as mesmas regras de decodificação do servidor (num teste, 37 de 39 saídas idênticas às do servidor) e conferindo cada
decisão com o validador do produto. Referência: modelo de 39 M treinado só na CPU com ~1 100 exemplos ⇒ 84% de decisões válidas.

In [ ]:
!mkdir -p model/pretrained && cp {WORK}/data/tokenizer.json model/pretrained/tokenizer.json
import time
REPORT = f'{WORK}/agent-eval-{PRESET}-{RUN}-{int(time.time())}.json'
!python pretrain/eval_agent_sft.py --checkpoint {WORK}/sft-{PRESET}-{RUN}/sft.safetensors --backend direct --device cuda --per-kind 30 --max-tokens 512 --report {REPORT}
print('relatório salvo em', REPORT)

## 5) Levar para o computador
Baixe `sft-<preset>-<run>/sft.safetensors` (+ `.json`) e `data/tokenizer.json`. No computador, copie o tokenizer para
`model/pretrained/tokenizer.json` (caminho que o checkpoint declara). A bancada de programação roda no computador, pelo servidor:

`python scripts/evaluate_programming_qualification.py --suite reserved --backend checkpoint --checkpoint model/pretrained/sft.safetensors --output <novo-relatorio>.json`

Referências (02-03/10, `small`): programação 0/96 aprovadas, 22 no contrato JSON; agente 0% de decisões válidas antes do treino agêntico.